# Experimento MAREA — Variante: **MAREA-bio**

**MAREA-bio — formulacion de tres fases biologicas (ciclo de Cobetia sp.)**

- Fase I  (planctonica)    — explotacion intensa guiada por mu~ (growth)
- Fase II (biofilm)        — exploracion estructurada con vecindad, si B~ > 0.45
- Fase III(desprendimiento)— salto de Levy desde el mejor global, con prob. p_det

### Que pregunta responde este notebook

MAREA-base quedo ultima o penultima en 5 de 6 configuraciones frente a los
competidores con presupuesto correcto (ABC, DE, GWO, PSO, WOA). Su unico
cuadrante competitivo es D50 / 500k FES.

Diagnostico: en MAREA-base los coeficientes alpha, beta y gamma dependen solo
de t/T. beta y gamma decaen a 0 y alpha sube a 0.99, asi que al final la
actualizacion es `X_i + 0.99*(X_g - X_i)`: la poblacion colapsa sobre el
incumbente sin ninguna fuente de diversidad.

**MAREA-bio nunca se corrio sobre CEC.** La parte bioinspirada — las tres
fases, el salto de Levy, los indicadores mu~/B~/V~ — no tiene evidencia ni a
favor ni en contra. Este notebook la produce.

### Protocolo experimental
- **Benchmarks:** 71 funciones CEC 2014 (30) + CEC 2017 (29) + CEC 2022 (12)
- **Dimensiones:** 10D y 50D (CEC2022: 10D y 20D)
- **MaxFES:** 5,000 / 50,000 / 500,000 / 5,000,000
- **Corridas:** 51 independientes por configuracion
- **Guardado:** incremental en JSON — se puede interrumpir y reanudar

> Ejecutar las celdas en orden. **La celda de verificacion de entorno no es
> opcional**: si el veredicto no es "ESTE ES EL ENTORNO CORRECTO", lo que se
> calcule aqui no es comparable con los competidores ya guardados.


In [ ]:
import sys, os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Asegurar que el directorio del proyecto este en el path
PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from experimento_config import (
    run_experiment, load_partial, CEC_REGISTRY,
    MAXFES, DIMS, N_RUNS, POP_SIZE, RESULTS_DIR
)

# ── Variante ──────────────────────────────────────────────────────────────
# sigma1/sigma2 multiplican xi ~ U(-1,1) en unidades ABSOLUTAS y se calibraron
# en dominios de ancho ~10.24 (Sphere/Rastrigin), donde sigma1=0.4 es el 3.9%
# del ancho. En CEC el dominio es [-100,100] y ese mismo 0.4 pasa a ser el
# 0.2%: el termino de ruido queda practicamente inerte. Notese que eta (Levy)
# SI esta escalado por (U-L) dentro del mismo algoritmo.
#   SIGMA_REL = False -> port fiel de marea_comparacion.py     (arma primaria)
#   SIGMA_REL = True  -> sigma reescalado al ancho del dominio (arma de control)
SIGMA_REL = False

ALG_NAME  = 'MAREABIO_REL' if SIGMA_REL else 'MAREABIO'

# El Mac mini tiene 24 cores. Dejar aire si hay otra corrida en curso
# (p. ej. el 5M de MAREA-base).
N_WORKERS = 8

# Guarda de seguridad: las celdas de 50k / 500k / 5M no se lanzan con "Run All".
# Ponlo en True solo cuando quieras comprometer horas de maquina.
RUN_HEAVY = False

from marea_bio_benchmark import marea_bio, MAREABioParams
p = MAREABioParams()
p.sigma_rel = SIGMA_REL

# La parada anticipada debe estar DESACTIVADA: el protocolo CEC exige agotar
# MaxFES. marea_comparacion.marea_bio paraba tras 50 iteraciones sin mejora.
assert p.early_stop is False, 'early_stop debe estar desactivado para el benchmark'

def alg_fn(f, bounds, N, T, seed):
    p.N = N; p.T = T
    return marea_bio(f, bounds, p, seed=seed)

print(f'\u2713 {ALG_NAME} cargado correctamente')
print(f'  sigma_rel  = {p.sigma_rel}')
print(f'  early_stop = {p.early_stop}  (agota MaxFES completo)')
print(f'  Resultados en: {RESULTS_DIR}/')
print(f'  Funciones CEC disponibles: {len(CEC_REGISTRY)}')
print(f'  Workers: {N_WORKERS} | RUN_HEAVY: {RUN_HEAVY}')


## 1. Verificacion de entorno — **no saltarse**

Re-ejecuta corridas guardadas de los competidores con las mismas semillas y
las compara bit a bit. Solo si dice **ESTE ES EL ENTORNO CORRECTO** los
resultados de MAREA-bio seran comparables con el resto del benchmark.

Se lanza con `sys.executable`, o sea el interprete real del kernel: cambiar el
kernel en Jupyter/VS Code no reinicia el proceso, asi que la seleccion del menu
no es prueba de nada.


In [ ]:
!{sys.executable} verificar_entorno.py


## 2. Smoke test + auditoria de presupuesto

Dos comprobaciones antes de comprometer maquina:

1. **Agota MaxFES** — ninguna corrida puede parar antes de T iteraciones.
2. **No se pasa de MaxFES** — FA gasta 40x y CLONALG 3.5x del presupuesto
   nominal porque el runner presupuesta en iteraciones, no en evaluaciones.
   MAREA-bio debe salir en 1.02x, igual que MAREA-base, PSO, DE, GWO, WOA y ABC.


In [ ]:
from experimento_config import get_problems_for_dim
import time

problems_10d = get_problems_for_dim(10)
test_prob    = problems_10d[0]   # F1 de CEC2014
print(f'Probando con: {test_prob["name"]} | dim={test_prob["dim"]}')

T_TEST = 62   # 62 iteraciones x 80 individuos = 4,960 FES ~ MaxFES=5,000

t0 = time.time()
for seed in range(3):
    r  = alg_fn(test_prob['f'], test_prob['bounds'], N=POP_SIZE, T=T_TEST, seed=seed)
    ok = 'OK' if r['iters'] == T_TEST else 'PARADA ANTICIPADA'
    print(f'  seed={seed}  best_f={r["best_f"]:.4e}  iters={r["iters"]}/{T_TEST}  [{ok}]')
    assert r['iters'] == T_TEST, 'Alguna corrida paro antes de agotar el presupuesto'
print(f'Tiempo: {time.time()-t0:.2f}s')

# ── Auditoria de presupuesto ──────────────────────────────────────────────
class _Contador:
    def __init__(self, f): self.f, self.n = f, 0
    def __call__(self, x):
        self.n += 1
        return self.f(x)

c = _Contador(test_prob['f'])
alg_fn(c, test_prob['bounds'], N=POP_SIZE, T=T_TEST, seed=0)
nominal = POP_SIZE * T_TEST
print(f'FES reales = {c.n:,}   nominal = {nominal:,}   ratio = {c.n/nominal:.2f}x')
assert c.n <= nominal + POP_SIZE, 'MAREA-bio se pasa del presupuesto asignado'
print('\u2713 Presupuesto agotado y respetado. Listo para el experimento.')


## 3. Diagnostico — se usan realmente las tres fases?

`p_det = p_det_base * V~ * (1 + B~)` gobierna la Fase III y `B~ > 0.45` la
Fase II. Si la poblacion colapsa rapido, B~ se dispara y el algoritmo se queda
atrapado en Fase II: la estructura de tres fases seria decorativa.

Esta celda es barata (6 corridas cortas) y contesta esa pregunta antes de
gastar horas.


In [ ]:
problems_50d = get_problems_for_dim(50)

print(f'{"problema":<12}{"dim":>5}{"Fase I":>12}{"Fase II":>12}{"Fase III":>12}')
print('-' * 53)
for probs, nd in ((problems_10d, 10), (problems_50d, 50)):
    for pid in (0, 35, 65):        # una funcion de CEC2014 / 2017 / 2022
        prob = probs[pid]
        pp = MAREABioParams(N=POP_SIZE, T=T_TEST, sigma_rel=SIGMA_REL)
        r  = marea_bio(prob['f'], prob['bounds'], pp, seed=0)
        tot = sum(r['phases']) or 1
        pct = [100 * x / tot for x in r['phases']]
        print(f'{prob["name"]:<12}{prob["dim"]:>5}'
              f'{pct[0]:>11.1f}%{pct[1]:>11.1f}%{pct[2]:>11.1f}%')
print('\nSi una fase se lleva >90% en todas las filas, las otras dos no estan')
print('aportando y el diseno de tres fases hay que revisarlo antes de publicarlo.')


## Experimento — MaxFES = 5,000  ← **ejecutar esta**
*Caso de uso: optimizacion costosa / tiempo-real*

Coste estimado: ~6 min con 8 workers (medido para MAREA-base).


In [ ]:
# ── MaxFES = 5,000 ──────────────────────────────
# Se corre sobre 10D y 50D. Cada corrida guarda su resultado.
# Si se interrumpe, la siguiente ejecucion retoma desde donde quedo.

for ndim in DIMS:
    run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=5000, n_workers=N_WORKERS)


## Experimento — MaxFES = 50,000  ← **y esta**
*Caso de uso: aplicaciones practicas estandar*

Coste estimado: ~17 min (D=10) + ~40 min (D=50) con 8 workers.

Estas dos configuraciones bastan para contestar si las fases biologicas
arreglan el colapso de diversidad. Las de 500k y 5M solo valen la pena
si el resultado a 5k/50k es alentador.


In [ ]:
# ── MaxFES = 50,000 ──────────────────────────────
# Se corre sobre 10D y 50D. Cada corrida guarda su resultado.
# Si se interrumpe, la siguiente ejecucion retoma desde donde quedo.

for ndim in DIMS:
    run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=50000, n_workers=N_WORKERS)


## Experimento — MaxFES = 500,000
*Caso de uso: convencion CEC estandar*

Coste estimado: ~2.8 h (D=10) + ~6.7 h (D=50) con 8 workers.


In [ ]:
# ── MaxFES = 500,000 ──────────────────────────────
# Coste: ~2.8 h (D=10) + ~6.7 h (D=50) con 8 workers
# Requiere RUN_HEAVY = True en la primera celda.

if RUN_HEAVY:
    for ndim in DIMS:
        run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=500000, n_workers=N_WORKERS)
else:
    print('Saltada (RUN_HEAVY = False). Coste: ~2.8 h (D=10) + ~6.7 h (D=50) con 8 workers')


## Experimento — MaxFES = 5,000,000
*Caso de uso: busqueda exploratoria profunda*

Coste estimado: ~10x el de 500k — noche completa de maquina.


In [ ]:
# ── MaxFES = 5,000,000 ──────────────────────────────
# Coste: ~10x el de 500k — noche completa de maquina
# Requiere RUN_HEAVY = True en la primera celda.

if RUN_HEAVY:
    for ndim in DIMS:
        run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=5000000, n_workers=N_WORKERS)
else:
    print('Saltada (RUN_HEAVY = False). Coste: ~10x el de 500k — noche completa de maquina')


## Resumen de resultados


In [ ]:
print(f'\n=== RESUMEN {ALG_NAME} ===')
print(f'{"Config":<30} {"Completadas":>12} {"Mean":>14} {"Best":>14}')
print('-' * 72)

for mf in MAXFES:
    for nd in DIMS:
        data = load_partial(ALG_NAME, nd, mf)
        vals = list(data.values())
        label = f'D={nd:2d} MaxFES={mf:>10,}'
        if vals:
            print(f'{label:<30} {len(vals):>12} {np.mean(vals):>14.4e} {np.min(vals):>14.4e}')
        else:
            print(f'{label:<30}   (sin resultados aun)')


## Comparacion — MAREA-bio contra MAREA-base y los competidores

La metrica es la **media de las 51 corridas** y los empates a menos de 1e-8
comparten rango, como pide el setup experimental (secciones 5 y 6).

Las dos columnas que importan:

- **bio vs base** — cuantas de las 71 funciones mejora la formulacion biologica.
  Es la pregunta que motiva el notebook.
- **rank medio** — donde queda MAREA-bio entre los competidores de presupuesto
  limpio. FA y CLONALG quedan fuera: gastan 40x y 3.5x su presupuesto, asi que
  sus numeros no son comparables hasta que se arregle el corte por FES.


In [ ]:
import json, glob
from scipy.stats import rankdata

LIMPIOS = ['ABC', 'DE', 'GWO', 'PSO', 'WOA']    # 1.02x del presupuesto

def _buscar(alg, nd, mf):
    """Los JSON de MAREA pueden estar en RESULTS_DIR o en la raiz del proyecto."""
    for d in (RESULTS_DIR, PROJECT_DIR, '.'):
        p = os.path.join(d, f'{alg}_D{nd}_FES{mf}.json')
        if os.path.exists(p):
            return p
    return None

def _load(alg, nd, mf):
    p = _buscar(alg, nd, mf)
    if p is None:
        return None
    M = np.full((71, N_RUNS), np.nan)
    for k, v in json.load(open(p)).items():
        a, b = k.split('_')
        M[int(a[3:]), int(b[3:])] = v
    return M if np.isfinite(M).mean() > 0.99 else None

def _ranks(mean_col, tol=1e-8):
    r = rankdata(mean_col); order = np.argsort(mean_col); out = r.copy()
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and abs(mean_col[order[j+1]] - mean_col[order[i]]) < tol:
            j += 1
        if j > i:
            avg = np.mean([r[order[k]] for k in range(i, j+1)])
            for k in range(i, j+1):
                out[order[k]] = avg
        i = j + 1
    return out

for nd in DIMS:
    for mf in MAXFES:
        bio = _load(ALG_NAME, nd, mf)
        if bio is None:
            continue
        base = _load('MAREA', nd, mf)
        print(f'\n=== D{nd}  MaxFES={mf:,} ' + '=' * 34)

        if base is not None:
            mb, mB = bio.mean(axis=1), base.mean(axis=1)
            gana = int((mb < mB).sum())
            print(f'  bio vs base : bio mejor en {gana}/71 funciones '
                  f'({gana/71:.0%})')
            for s, sl in (('CEC2014', slice(0,30)), ('CEC2017', slice(30,59)),
                          ('CEC2022', slice(59,71))):
                g = int((mb[sl] < mB[sl]).sum()); n = len(mb[sl])
                print(f'      {s}: {g}/{n}')
        else:
            print('  bio vs base : (faltan resultados de MAREA-base)')

        pool = {ALG_NAME: bio}
        if base is not None:
            pool['MAREA'] = base
        for a in LIMPIOS:
            m = _load(a, nd, mf)
            if m is not None:
                pool[a] = m
        if len(pool) < 3:
            continue
        names = list(pool)
        means = np.array([pool[a].mean(axis=1) for a in names])
        R  = np.column_stack([_ranks(means[:, q]) for q in range(71)])
        mr = R.mean(axis=1)
        orden = np.argsort(mr)
        pos = [names[i] for i in orden].index(ALG_NAME) + 1
        print(f'  rank medio  : {ALG_NAME} {pos}o de {len(names)}')
        print('      ' + '  '.join(f'{names[i]}:{mr[i]:.2f}' for i in orden))
